In [1]:
import mne
from mne.datasets import eegbci
import numpy as np
import os

In [2]:
# 1. Setup
subjects = range(1, 21) 
runs = [3, 7, 11]       

# TEAMMATE FIX 1: Defensive assertion to protect the binary mapping
assert all(r in [3, 7, 11] for r in runs), "ERROR: Binary mapping only valid for physical movement runs (3, 7, 11)!"

all_epochs = []
labels_identity = [] 
labels_utility = []  
run_tracker = []

In [3]:
print("Starting the bulletproof local pipeline...")

for subject in subjects:
    print(f"Processing Subject {subject:03d}...")
    
    # TEAMMATE FIX 2: Move the processing INSIDE the run loop
    for run in runs:
        file_path = f"dataset/S{subject:03d}/S{subject:03d}R{run:02d}.edf"
        
        if not os.path.exists(file_path):
            print(f"  WARNING: Could not find {file_path}. Skipping.")
            continue
            
        raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        
        # Clean and filter
        eegbci.standardize(raw)
        montage = mne.channels.make_standard_montage('standard_1005')
        raw.set_montage(montage)
        raw.filter(1., 50., fir_design='firwin', skip_by_annotation='edge', verbose=False)
        
        # Events
        custom_mapping = {'T0': 0, 'T1': 1, 'T2': 2}
        events, _ = mne.events_from_annotations(raw, event_id=custom_mapping, verbose=False)
        
        # TEAMMATE FIX 3: Force exactly 160 samples per epoch
        tmin = 0.0
        tmax = 159.0 / 160.0 
        epochs = mne.Epochs(raw, events, event_id=custom_mapping, tmin=tmin, tmax=tmax, 
                            proj=False, baseline=None, preload=True, verbose=False)
        
        # TEAMMATE FIX 4: Convert Volts to Microvolts for CNN stability
        data = epochs.get_data(copy=False) * 1e6 
        
        # Tag every epoch individually at the source
        for i in range(len(data)):
            task_code = epochs.events[i, 2]
            binary_task = 1 if task_code in [1, 2] else 0
            
            all_epochs.append(data[i])
            labels_identity.append(subject - 1)
            labels_utility.append(binary_task)
            # The run label is assigned at load time, perfectly safe!
            run_tracker.append(run)

Starting the bulletproof local pipeline...
Processing Subject 001...
Processing Subject 002...
Processing Subject 003...
Processing Subject 004...
Processing Subject 005...
Processing Subject 006...
Processing Subject 007...
Processing Subject 008...
Processing Subject 009...
Processing Subject 010...
Processing Subject 011...
Processing Subject 012...
Processing Subject 013...
Processing Subject 014...
Processing Subject 015...
Processing Subject 016...
Processing Subject 017...
Processing Subject 018...
Processing Subject 019...
Processing Subject 020...


In [4]:
X = np.array(all_epochs)
Y_identity = np.array(labels_identity)
Y_utility = np.array(labels_utility)
Run_IDs = np.array(run_tracker)

print("\n--- Pipeline Complete ---")
print(f"Total ML Tensor Shape (X): {X.shape}") # Should end in exactly 160!
print(f"Identity Labels: {Y_identity.shape}")
print(f"Utility Labels: {Y_utility.shape}")
print(f"Run Tracker: {Run_IDs.shape}")


--- Pipeline Complete ---
Total ML Tensor Shape (X): (1800, 64, 160)
Identity Labels: (1800,)
Utility Labels: (1800,)
Run Tracker: (1800,)


In [5]:
# ASSUMPTION: 'raw' is still in memory from the final iteration of the pipeline loop.
# This is mathematically safe because our pipeline applies the standard 10-05 montage 
# to every file, guaranteeing all 20 subjects share the exact same 64-channel layout.
channel_names = raw.ch_names

# Standard Emotiv EPOC+ 10-10 locations
emotiv_targets = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

# Standard Muse locations (using T7/T8 as temporal proxies for TP9/TP10)
muse_targets = ['AF7', 'AF8', 'T7', 'T8']

# TEAMMATE FIX: Hard failure if channels are missing
missing_emotiv = [ch for ch in emotiv_targets if ch not in channel_names]
missing_muse = [ch for ch in muse_targets if ch not in channel_names]

if missing_emotiv:
    raise ValueError(f"Missing Emotiv channels: {missing_emotiv}")
if missing_muse:
    raise ValueError(f"Missing Muse channels: {missing_muse}")

emotiv_indices = [channel_names.index(ch) for ch in emotiv_targets]
muse_indices = [channel_names.index(ch) for ch in muse_targets]

print(f"Emotiv channel count: {len(emotiv_indices)} (expected 14)")
print(f"Muse channel count: {len(muse_indices)} (expected 4)")
print(f"Emotiv channels confirmed: {[channel_names[i] for i in emotiv_indices]}")
print(f"Muse channels confirmed: {[channel_names[i] for i in muse_indices]}")

Emotiv channel count: 14 (expected 14)
Muse channel count: 4 (expected 4)
Emotiv channels confirmed: ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
Muse channels confirmed: ['AF7', 'AF8', 'T7', 'T8']


In [6]:
save_dir = "processed_data"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, 'X_normalized.npy'), X)
np.save(os.path.join(save_dir, 'Y_identity.npy'), Y_identity)
np.save(os.path.join(save_dir, 'Y_utility.npy'), Y_utility)
np.save(os.path.join(save_dir, 'Run_IDs.npy'), Run_IDs)

# Save the channel masks
np.save(os.path.join(save_dir, 'emotiv_indices.npy'), np.array(emotiv_indices))
np.save(os.path.join(save_dir, 'muse_indices.npy'), np.array(muse_indices))

print(f"Checkpoint created: All arrays safely stored in the '{save_dir}/' folder.")

Checkpoint created: All arrays safely stored in the 'processed_data/' folder.


In [7]:
print(channel_names)

['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'O1', 'Oz', 'O2', 'Iz']
